# Part 3 — Gradual semantics
## Question 1: weighted h-categorizer

To answer this question, we first write a function `attack_matrix` that stores a graph as a matrix (`M[i, j] = 1` if $i$ attacks $j$). Then `weighted_h_categorizer` uses this matrix to apply the h-categorizer formula again and again, until the degrees stop moving (by less than $\varepsilon$). Both functions are in `hcategorizer.py`.

Then we test our function in three ways:
1. on a few graphs where we computed the degrees by hand: some from TD3, some with weights, and some special cases;
2. on convergence: how $\varepsilon$ changes the number of iterations;
3. with many weight vectors at once, as we will need in Question 2.

In [1]:
import numpy as np
from hcategorizer import attack_matrix, weighted_h_categorizer
from graphs import TEST_GRAPHS

### Test 1: does it give the right degrees?

For each graph we computed the degrees by hand (the details are in `graphs.py`). When no weights are given, all the weights are 1, like in TD3.

In [2]:
for name, g in TEST_GRAPHS.items():
    M = attack_matrix(g["arguments"], g["attacks"])
    # weights: all 1 if none are given
    if g["weights"] is None:
        w = np.ones(len(g["arguments"]))
    else:
        w = [g["weights"][a] for a in g["arguments"]]
    degrees, iterations = weighted_h_categorizer(M, w)
    # compare with the degrees computed by hand
    hand = [g["expected"][a] for a in g["arguments"]]
    print(f"{name}: {'OK' if np.allclose(degrees, hand) else 'MISMATCH'}")
    print("   ", dict(zip(g["arguments"], degrees.round(4).tolist())))

TD3 G1, one attack: OK
    {'p': 1.0, 'a': 0.5}
TD3 G2, three attackers: OK
    {'q': 1.0, 'r': 1.0, 's': 1.0, 'b': 0.25}
TD3 G4, defence: OK
    {'t': 1.0, 'p': 0.5, 'q': 0.5, 'a': 0.5}
TD3 G3 with weights: OK
    {'p': 0.6, 'q': 0.5625, 'a': 0.32, 'b': 0.3}
TD3 G4 with weights: OK
    {'t': 0.5, 'p': 0.5333, 'q': 0.2667, 'a': 0.5}
no attack: OK
    {'a': 0.3, 'b': 0.7}
attacker with weight 0: OK
    {'a': 0.8, 'b': 0.0}
a attacks itself: OK
    {'a': 0.618}


- The function gives the same degrees as by hand for every graph.
- The TD3 graphs also give the TD3 ranking: for example in G4, t = 1, then p, q and a are tied at 0.5.
- With weights, the result changes: in G4, p, q and a are not tied anymore (p = 0.53 > a = 0.5 > q = 0.27).
- The special cases behave as expected: without attack the degree is the weight, an attacker with weight 0 has no effect, and an argument that attacks itself never goes above 0.618.

### Test 2: convergence and $\varepsilon$

Graphs without cycles converge in a few iterations. With a cycle, the degrees go back and forth before they settle. We take $a \leftrightarrow b$ with weights 1, where the limit is known: $x = 1/(1+x)$, so $x = (\sqrt5-1)/2 \approx 0.618$.

In [3]:
M = attack_matrix(["a", "b"], [("a", "b"), ("b", "a")])
exact = (np.sqrt(5) - 1) / 2
for eps in [1e-2, 1e-4, 1e-6, 1e-9]:
    degrees, iterations = weighted_h_categorizer(M, [1, 1], epsilon=eps)
    print(f"eps = {eps:.0e}: {iterations} iterations, error = {abs(degrees[0] - exact):.1e}")

eps = 1e-02: 5 iterations, error = 2.6e-03
eps = 1e-04: 10 iterations, error = 2.2e-05
eps = 1e-06: 15 iterations, error = 1.8e-07
eps = 1e-09: 22 iterations, error = 2.1e-10


- A smaller $\varepsilon$ gives a more precise result, but needs more iterations.
- The error always stays below $\varepsilon$. We keep $\varepsilon = 10^{-6}$ by default.

### Test 3: many weight vectors at once

For Question 2 we need 100,000 weight vectors. So the function also accepts a table of weights (one row = one weight vector) and computes them all at once. Do we get the same result as one by one?

In [4]:
M = attack_matrix(["a", "b"], [("a", "b"), ("b", "a")])
W = np.random.default_rng(0).random((100_000, 2))  # 100,000 weight vectors, one per row

# 1) all the weight vectors at once (like in Question 2)
all_at_once, _ = weighted_h_categorizer(M, W)
# 2) one weight vector at a time (like in tests 1 and 2)
one_by_one = np.array([weighted_h_categorizer(M, w)[0] for w in W])

# look at the first 3 weight vectors
for i in range(3):
    print(f"weights {W[i].round(3)} -> at once: {all_at_once[i].round(4)} | one by one: {one_by_one[i].round(4)}")
print()
print("number of weight vectors:", len(W))
print("largest difference over the 100,000:", np.abs(all_at_once - one_by_one).max())

weights [0.637 0.27 ] -> at once: [0.5421 0.1749] | one by one: [0.5421 0.1749]
weights [0.041 0.017] -> at once: [0.0403 0.0159] | one by one: [0.0403 0.0159]
weights [0.813 0.913] -> at once: [0.5064 0.6059] | one by one: [0.5064 0.6059]

number of weight vectors: 100000
largest difference over the 100,000: 4.794045719291873e-07


- For each weight vector, we get the same degrees whether we compute them all at once or one by one.
- Over the 100,000 vectors, the largest difference is about $5 \times 10^{-7}$, below $\varepsilon$: the two ways give the same result. The function is ready for Question 2.